# Churn XAI — Notebook d'exploration

Projet Challenge 1 — Classification Explicable. Cas métier : **Télécom — Attrition client** (Telco Customer Churn).

Ce notebook reproduit le pipeline complet de l'application Streamlit (`app.py`) de façon linéaire,
en réutilisant les mêmes modules (`src/`), pour une inspection rapide sans lancer l'UI.

Pré-requis : exécuter depuis la racine `churn-xai-app/` avec l'environnement virtuel activé
(`pip install -r requirements.txt`).

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import plotly.express as px

from src import data as data_mod
from src import prep as prep_mod
from src import models as models_mod
from src import evaluate as eval_mod
from src import explain as explain_mod

## Module 1 — Données

In [ ]:
df = data_mod.download_telco_dataset()
df = data_mod.clean_telco(df)
print(df.shape)
df.head()

In [ ]:
target_col = data_mod.DEFAULT_TARGET
data_mod.class_balance(df, target_col)

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
px.histogram(df, x='tenure', color=target_col, barmode='overlay', nbins=30,
             title='Distribution de tenure par churn')

In [ ]:
numeric_cols, categorical_cols = prep_mod.split_feature_types(df.drop(columns=[target_col]))
corr = df[numeric_cols + [target_col]].corr(numeric_only=True)
px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Corrélations')

## Module 2 — Préparation

In [ ]:
feature_cols = [c for c in df.columns if c not in [target_col, 'customerID']]

work_df = df.dropna(subset=[target_col]).copy()
X = work_df[feature_cols]
y = work_df[target_col].astype(int)

numeric_cols, categorical_cols = prep_mod.split_feature_types(X)
preprocessor = prep_mod.build_preprocessor(numeric_cols, categorical_cols, scale=True)

X_train, X_test, y_train, y_test = prep_mod.make_split(X, y, test_size=0.2, stratify=True)
print(f"train={len(X_train)}  test={len(X_test)}  num={len(numeric_cols)}  cat={len(categorical_cols)}")

## Module 3 — Modélisation et comparaison

In [ ]:
trained = {}
for name in models_mod.MODEL_REGISTRY:
    params = models_mod.MODEL_REGISTRY[name]['default_params']
    pipeline, used_params = models_mod.train_model(
        name, preprocessor, X_train, y_train, params, optimize=False
    )
    trained[name] = pipeline
    print(name, '->', used_params)

## Module 4 — Évaluation

In [ ]:
models_proba = {name: p.predict_proba(X_test)[:, 1] for name, p in trained.items()}
metrics = eval_mod.metrics_table(models_proba, y_test, threshold=0.5)
metrics.sort_values('AUC-ROC', ascending=False)

In [ ]:
eval_mod.roc_overlay_figure(models_proba, y_test)

In [ ]:
best_model_name = metrics['AUC-ROC'].idxmax()
best_pipeline = trained[best_model_name]
print('Meilleur modèle :', best_model_name)
eval_mod.confusion_matrix_figure(y_test, models_proba[best_model_name], threshold=0.5)

## Module 5 — Explainable AI (SHAP / LIME)

Explications calculées directement ici (sans passer par l'API FastAPI, pour la version notebook).

In [ ]:
feature_names = explain_mod.get_transformed_feature_names(best_pipeline)
native_imp = explain_mod.native_importance(best_pipeline, feature_names)
native_imp.head(15)

In [ ]:
perm_imp = explain_mod.permutation_importances(best_pipeline, X_test, y_test, n_repeats=5)
perm_imp.head(15)

In [ ]:
shap_explainer = explain_mod.build_shap_explainer(best_pipeline, X_train)
shap_sample = X_test.sample(150, random_state=42)
shap_out = explain_mod.shap_values_for(shap_explainer, best_pipeline, shap_sample)
shap_out.values.shape

In [ ]:
# Explication locale pour un individu du jeu de test
row = X_test.iloc[0]
row_df = X_test.iloc[[0]]
proba = best_pipeline.predict_proba(row_df)[0, 1]

local_shap_out = explain_mod.shap_values_for(shap_explainer, best_pipeline, row_df)
shap_row = pd.Series(local_shap_out.values[0], index=feature_names)

lime_bundle = explain_mod.build_lime_explainer(
    X_train, numeric_cols, categorical_cols, class_names=['No', 'Yes']
)
lime_dict = explain_mod.lime_explanation_for(lime_bundle, best_pipeline, row)

synthesis = explain_mod.synthesis_sentence(shap_row, proba, threshold=0.5)
print(synthesis)
print('\nTop SHAP:', shap_row.abs().sort_values(ascending=False).head(5).index.tolist())
print('Top LIME:', list(lime_dict.keys())[:5])

## Conclusion

Voir `README.md` et `one_pager.pdf` à la racine du dépôt pour l'analyse critique complète
(cohérence SHAP/LIME, variables sensibles, compromis performance/interprétabilité, recommandations).